# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset on adoption predictors of indigenous and modern knowledge in rangeland management practices, Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import matplotlib.pyplot as plt
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("\n=== Dataset Metadata Overview ===")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review the available record sets, fields, and their IDs. All references are made by `@id` fields as per FAIR^2 standards.

In [ ]:
print("\n=== Record Sets Overview ===")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    {field.name} (@id: {field.id}, DataType: {field.data_type})")
    print()

# Show example records from the first available RecordSet
if record_sets:
    first_rs = record_sets[0]
    print(f"Sample records from RecordSet '{first_rs.name}' (@id: {first_rs.id}):")
    for x in dataset.records(record_set=first_rs.id):
        pprint.pprint(x)
        break  # Show only the first record for brevity
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from the available record sets into pandas DataFrames for analysis. All record sets and fields are referenced by their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records_iterator = dataset.records(record_set=record_set_id)
    records = list(records_iterator)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show the columns for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in RecordSet '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record set data available.")

## 4. Exploratory Data Analysis (EDA)
Process and explore the tabular data loaded from the record sets. Apply filtering, normalization, and grouping. All column references use their `@id`.

In [ ]:
# Choose a numeric field for EDA (example selection based on available fields)
# This will automatically choose the first numeric field from the first record set

numeric_field_id = None
group_field_id = None
first_rs = record_sets[0] if record_sets else None

if first_rs:
    # Try to guess numeric and group fields
    for field in first_rs.fields:
        if field.data_type in ('schema:Float', 'schema:Integer', 'Float', 'Integer') and not numeric_field_id:
            numeric_field_id = field.id
        if field.data_type in ('schema:Text', 'Text') and not group_field_id:
            group_field_id = field.id

    df = dataframes[first_rs.id]

    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by chosen group field
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Refer to all fields by their `@id` for consistency.

In [ ]:
# Example: Visualize the distribution of the selected numeric field
if first_rs and numeric_field_id and numeric_field_id in dataframes[first_rs.id].columns:
    df = dataframes[first_rs.id]
    plt.figure(figsize=(8, 5))
    plt.hist(df[numeric_field_id].dropna(), bins=30, alpha=0.7)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

    # If there is also a group field, show a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 6))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration, referencing data elements by their `@id` as per best FAIR^2 practices. 

The FAIR^2 dataset provides a structured overview of predictors of knowledge adoption among pastoralist households, with explicit field and column `@id`s for traceability. Exploration enables filtering, normalization, and visualization of key numeric factors (such as log likelihood, coefficients, etc.) grouped by demographic attributes, supporting further policy and academic analysis.